# Final Project Operations
**Business 35137 — Winter 2026**

In [2]:
from edgar import Company, set_identity
import pandas as pd
import re
import os
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import pgeocode

# Suppress noisy edgar parser messages
logging.getLogger('edgar').setLevel(logging.ERROR)

DATA_DIR = '../Data/'
intermediate_dir = os.path.join(DATA_DIR, 'Intermediate')

In [3]:
intermediate_path = os.path.join(intermediate_dir, 'master_crsp_merged.parquet')
master_crsp = pd.read_parquet(intermediate_path)

In [4]:
set_identity(os.environ.get("SEC_EDGAR_IDENTITY", "Your Name your.email@example.com"))  # SEC requires a name + email

def clean_tic(t):
    return re.sub(r'\.\d*$', '', str(t)).rstrip('.')

# ── 0. Filter to Russell 2000 / S&P 500 universe ──────────────────────────────
universe = master_crsp[master_crsp['index'].isin(['Russell 2000', 'S&P 500'])]

# ── Filter to atmospheric sectors only ────────────────────────────────────────
ATMOS_SECTORS = ['Agriculture & Food', 'Utilities & Telecom', 'Leisure & Tourism']
universe = universe[universe['atmos_sector'].isin(ATMOS_SECTORS)]
print(f"Sector breakdown:\n{universe['atmos_sector'].value_counts()}")

print(f"\nUniverse: {universe['permno'].nunique():,} unique permnos")
print(f"  Russell 2000 rows: {(universe['index'] == 'Russell 2000').sum():,}")
print(f"  S&P 500 rows:      {(universe['index'] == 'S&P 500').sum():,}")

# ── 1. Build gvkey / ticker mappings ──────────────────────────────────────────
_map = (
    universe[['gvkey', 'tic']].dropna(subset=['gvkey', 'tic'])
    .drop_duplicates('tic')
)
_map['clean'] = _map['tic'].astype(str).map(clean_tic)
ticker_to_gvkey: dict[str, str] = _map.set_index('clean')['gvkey'].to_dict()

# ── 2. Tickers + first/last index year per ticker ─────────────────────────────
_tic_yr = (
    universe[['tic', 'date']].dropna(subset=['tic'])
    .assign(
        clean=lambda d: d['tic'].astype(str).map(clean_tic),
        year=lambda d: d['date'].dt.year,
    )
)
_tic_yr = _tic_yr[_tic_yr['clean'].str[0:1].str.isalpha()]

ticker_first: dict[str, int] = _tic_yr.groupby('clean')['year'].min().to_dict()
ticker_last:  dict[str, int] = _tic_yr.groupby('clean')['year'].max().to_dict()

edgar_to_try = sorted({
    c for t in universe[universe['date'] >= '1993-01-01']['tic'].dropna().unique()
    if (c := clean_tic(t)) and c[0].isalpha()
})
print(f"\nUnique EDGAR candidates (>=1993): {len(edgar_to_try):,}")

# ── 3. State maps ─────────────────────────────────────────────────────────────
us_states_map = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}
US_STATE_ABBRS = set(us_states_map.values()) | {"DC"}

# ── 4. City bank: {state_abbr -> frozenset of known lowercase city names} ──────
def _build_city_bank() -> dict[str, frozenset[str]]:
    nomi = pgeocode.Nominatim('us')
    df = nomi._data[['place_name', 'state_code']].dropna()
    bank = {}
    for state, grp in df.groupby('state_code'):
        bank[state] = frozenset(grp['place_name'].str.lower())
    return bank

CITY_BANK = _build_city_bank()
print(f"City bank: {sum(len(v) for v in CITY_BANK.values()):,} city-state entries across {len(CITY_BANK)} states")

# ── 5. City/state/zip extraction — validated against CITY_BANK ────────────────
_CITY_ABBR_RE = re.compile(
    r'\b'
    r'((?:[A-Z][a-zA-Z\-\'\.]+)(?:\s+[A-Z][a-zA-Z\-\'\.]+){0,3})'
    r',\s*([A-Z]{2})'
    r'(?:\s+(\d{5}(?:-\d{4})?))?'
    r'(?=[\s,;.\)\n]|$)'
)
_STATE_NAMES_PATTERN = '|'.join(
    re.escape(s) for s in sorted(us_states_map, key=len, reverse=True)
)
_CITY_FULLSTATE_RE = re.compile(
    r'\b'
    r'((?:[A-Z][a-zA-Z\-\'\.]+)(?:\s+[A-Z][a-zA-Z\-\'\.]+){0,3})'
    r',\s*(' + _STATE_NAMES_PATTERN + r')'
    r'\b'
)

def extract_locations(text: str) -> list[tuple]:
    text = re.sub(r'\s+', ' ', text)
    best: dict[tuple, str | None] = {}
    for m in _CITY_ABBR_RE.finditer(text):
        city, state, z = m.group(1).strip(), m.group(2), m.group(3)
        if state in US_STATE_ABBRS and city.lower() in CITY_BANK.get(state, frozenset()):
            k = (city, state)
            if k not in best or (z is not None and best[k] is None):
                best[k] = z
    for m in _CITY_FULLSTATE_RE.finditer(text):
        city  = m.group(1).strip()
        state = us_states_map[m.group(2)]
        if city.lower() in CITY_BANK.get(state, frozenset()):
            k = (city, state)
            if k not in best:
                best[k] = None
    return [(city, state, z) for (city, state), z in best.items()]

def get_locations_from_filing(filing) -> tuple[list[tuple], object]:
    tenk_obj = filing.obj()
    if 'Item 2' not in tenk_obj.items:
        return [], filing.filing_date
    raw = tenk_obj['Item 2']
    return extract_locations(raw), filing.filing_date

# ── 6. Worker ─────────────────────────────────────────────────────────────────
# Accept any 10-K filed within the company's CRSP tenure [first_year, last_year].
# Walk newest→oldest and use the first filing that contains Item 2.
def fetch_edgar(ticker):
    try:
        first_year = ticker_first.get(ticker)
        last_year  = ticker_last.get(ticker)
        gvkey      = ticker_to_gvkey.get(ticker)
        if first_year is None or last_year is None:
            return ticker, []

        company = Company(ticker)
        filings = company.get_filings(form="10-K")
        if not filings:
            return ticker, []

        # Accept any 10-K filed during the company's tenure
        windowed = [f for f in filings
                    if first_year <= f.filing_date.year <= last_year]
        if not windowed:
            return ticker, []

        # Walk newest→oldest; stop at the first filing that has Item 2
        chosen = None
        for f in windowed:
            try:
                if 'Item 2' in f.obj().items:
                    chosen = f
                    break
            except Exception:
                continue
        if chosen is None:
            return ticker, []

        rows = []
        try:
            locs, fdate = get_locations_from_filing(chosen)
            filing_zip: dict[tuple, str | None] = {}
            for city, state, z in locs:
                k = (city, state)
                if k not in filing_zip or (z is not None and filing_zip[k] is None):
                    filing_zip[k] = z
            for (city, state) in sorted(filing_zip):
                rows.append({
                    "gvkey":       gvkey,
                    "Ticker":      ticker,
                    "First_Year":  first_year,
                    "Last_Year":   last_year,
                    "Filing_Date": fdate,
                    "City":        city,
                    "State":       state,
                    "Zip":         filing_zip[(city, state)],
                })
        except Exception:
            pass

        return ticker, rows

    except Exception:
        return ticker, []

# ── 7. Parallel EDGAR calls ───────────────────────────────────────────────────
results = []
n       = len(edgar_to_try)

print(f"\nFetching 10-Ks for {n:,} tickers (8 workers)...")
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(fetch_edgar, t): t for t in edgar_to_try}
    for i, future in enumerate(as_completed(futures), 1):
        ticker, rows = future.result()
        if rows:
            results.extend(rows)
        if i % 100 == 0 or i == n:
            print(f"[{i:,}/{n:,}] processed | {len(results):,} rows collected")

total_tickers = len({r["Ticker"] for r in results})
print(f"\nDone. {total_tickers:,} tickers | {len(results):,} total rows")

out = pd.DataFrame(results)
out.to_csv(os.path.join(intermediate_dir, "Industry_SP_Russell_Company_Operations.csv"), index=False)

Sector breakdown:
atmos_sector
Utilities & Telecom    106993
Agriculture & Food      56943
Leisure & Tourism       51406
Name: count, dtype: int64

Universe: 2,087 unique permnos
  Russell 2000 rows: 135,013
  S&P 500 rows:      80,329

Unique EDGAR candidates (>=1993): 1,136
City bank: 29,542 city-state entries across 52 states

Fetching 10-Ks for 1,136 tickers (8 workers)...
[100/1,136] processed | 48 rows collected
[200/1,136] processed | 91 rows collected
[300/1,136] processed | 171 rows collected
[400/1,136] processed | 278 rows collected
[500/1,136] processed | 334 rows collected
[600/1,136] processed | 421 rows collected
[700/1,136] processed | 475 rows collected
[800/1,136] processed | 578 rows collected
[900/1,136] processed | 676 rows collected
[1,000/1,136] processed | 758 rows collected
[1,100/1,136] processed | 826 rows collected
[1,136/1,136] processed | 846 rows collected

Done. 209 tickers | 846 total rows
